# Explore the built data

Loads `docs/data.json` (run `python scraper/build.py` first) into a DataFrame with the same metrics as the site.

In [ ]:
import json

import pandas as pd

with open("../docs/data.json", encoding="utf-8") as fh:
    data = json.load(fh)

df = pd.DataFrame(data["rows"], columns=data["columns"])
df["dept"] = df["dept"].map(lambda i: data["departments"][i])
df["aisle"] = df["aisle"].map(lambda i: data["aisles"][i])
df["promo"] = df["promo"].map(lambda i: data["promos"][int(i)] if pd.notna(i) else None)
flag_bits = data["flags"]
df["flag_names"] = df["flags"].map(lambda f: [name for name, bit in flag_bits.items() if f & bit])

df["share"] = 4 * df["protein"] / df["kcal"]    # protein share of calories
df["gpp"] = 10 * df["protein"] / df["ppk"]       # g protein per £, shelf price
df["cc_gpp"] = 10 * df["protein"] / df["ccppk"]  # g protein per £, Clubcard price
print(f"{len(df)} products, data as of {data['scraped']}")

Top products with the site's default filters: no suspect data, a price per kg, and at least 25% of calories from protein.

In [ ]:
suspect = flag_bits["bad_macros"] | flag_bits["bad_kcal"] | flag_bits["bad_price"]
ranked = df[((df["flags"] & suspect) == 0) & df["gpp"].notna() & (df["share"] >= 0.25)]
columns = ["title", "aisle", "price", "kg", "protein", "kcal", "share", "gpp", "cc_gpp", "flag_names"]
ranked.sort_values("gpp", ascending=False)[columns].head(30)

Filter by aisle, e.g. every egg pack:

In [ ]:
eggs = ranked[ranked["aisle"] == "Eggs"]
eggs.sort_values("gpp", ascending=False)[columns]